In [16]:
import pandas as pd
import re
from deep_translator import GoogleTranslator

In [40]:
pd.set_option('display.max_rows', None)

## MAS-IT dataframes merge

In [2]:
rassereno_path = '/Users/emma/projects/cognoscolab/rassereno'
math_ma_llm = '/Users/emma/projects/cognoscolab/math_anxiety_LLMs'

In [3]:
mas_it_2023 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2023/mas_it_2023_humans.csv')
mas_it_humans_2024 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2024/mas_it_2024_humans.csv')
mas_it_BA_2024 = pd.read_csv(f'{rassereno_path}/out/formatting/psychology_BA_2024/mas_it_2024.csv')
mas_it_2025 = pd.read_csv(f'{rassereno_path}/out/formatting/all_students_2025/mas_it_2025.csv')

In [4]:
mas_it_university_2025 = mas_it_2025[
    mas_it_2025['participant_id'].str.startswith('interfacce') |
    mas_it_2025['participant_id'].str.startswith('fisica')
].reset_index(drop=True)

mas_it_liceo_2025 = mas_it_2025[
    mas_it_2025['participant_id'].str.startswith('liceo')
].reset_index(drop=True)

mas_it_liceo_2025.to_csv(f'{rassereno_path}/out/formatting/liceo_2025/mas_it_liceo_2025.csv', index=False)

In [5]:
# get the column names of the 2025 dataset
cols_2025 = mas_it_university_2025.columns

# restrict all dataframes to those columns before concatenation
dfs = [
    mas_it_humans_2024[cols_2025.intersection(mas_it_humans_2024.columns)],
    mas_it_2023[cols_2025.intersection(mas_it_2023.columns)],
    mas_it_BA_2024[cols_2025.intersection(mas_it_BA_2024.columns)],
    mas_it_university_2025[cols_2025]
]

# concatenate them
all_universities_mas_it = pd.concat(dfs, ignore_index=True)
all_universities_mas_it.to_csv(f'{rassereno_path}/out/formatting/all_university_students/mas_it_all_universities.csv', index=False)

## Associations and Valence dataframes merge

In [6]:
associations_valence_2023 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2023/associations_valence_2023_humans.csv')
associations_valence_2024 = pd.read_csv(f'{math_ma_llm}/out/formatting/humans_2024/associations_valence_2024_humans.csv')
associations_valence_BA_2024 = pd.read_csv(f'{rassereno_path}/out/formatting/psychology_BA_2024/associations_valence_2024.csv')
associations_valence_2025 = pd.read_csv(f'{rassereno_path}/out/formatting/all_students_2025/associations_valence_task_2025.csv')

In [7]:
associations_valence_university_2025 = associations_valence_2025[
    associations_valence_2025['participant_id'].str.startswith('interfacce') |
    associations_valence_2025['participant_id'].str.startswith('fisica')
]
associations_valence_liceo_2025 = associations_valence_2025[
    associations_valence_2025['participant_id'].str.startswith('liceo')
]
associations_valence_liceo_2025.to_csv(f'{rassereno_path}/out/formatting/liceo_2025/associations_valence_liceo_2025.csv', index=False)

In [8]:
all_universities_associations_valence = pd.concat([associations_valence_2024, associations_valence_2023, associations_valence_BA_2024, associations_valence_university_2025], ignore_index=True)
all_universities_associations_valence.to_csv(f'{rassereno_path}/out/formatting/all_university_students/associations_valence_all_universities.csv', index=False)

## Common words of multiple datasets

In [9]:
common_words_university = list(set(associations_valence_2024['cue_word']) & set(associations_valence_2023['cue_word']) & set(associations_valence_BA_2024['cue_word']) & set(associations_valence_university_2025['cue_word']))

print(f'Common word of all university subgroups: {common_words_university}')

Common word of all university subgroups: ['sfida', 'esame', 'equazione', 'fisica', 'matematica', 'ansia', 'statistica']


In [10]:
gpt_4o_associations_valence = pd.read_csv('/Users/emma/projects/cognoscolab/math_anxiety_LLMs/out/formatting/gpt_4o/associations_valence_gpt_4o.csv')

In [11]:
list(set(associations_valence_liceo_2025['cue_word']) & set(all_universities_associations_valence['cue_word']) & set(gpt_4o_associations_valence['cue_word']))

['sfida',
 'scienza',
 'esame',
 'equazione',
 'fisica',
 'matematica',
 'ansia',
 'informatica',
 'statistica',
 'innovazione',
 'scienziato',
 'benessere']

In [12]:
def cue_word_extraction(file):
    data = open(file).readlines()

    pairs = [re.findall(r'"([^"]+)"', line) for line in data]

    df = pd.DataFrame(pairs, columns=['cue_word', 'word'])

    counts = df['cue_word'].value_counts()

    cue_words = counts[:len(counts) // 2]

    cue_words_list = cue_words.index.tolist()

    return df, cue_words_list

In [25]:
experts_df, cue_words_experts = cue_word_extraction('/Users/emma/Desktop/data_2021/Experts_unfiltered.txt')

In [14]:
students_df, cue_words_students = cue_word_extraction('/Users/emma/Desktop/data_2021/Students_unFiltered.txt')

In [15]:
list(set(associations_valence_liceo_2025['cue_word']) & set(all_universities_associations_valence['cue_word']) & set(gpt_4o_associations_valence['cue_word']) & set(cue_words_students))

['fisica', 'matematica', 'esame', 'scienziato']

In [26]:
translated = [GoogleTranslator(source='en', target='it').translate(word) for word in cue_words_experts]

In [27]:
list(set(associations_valence_liceo_2025['cue_word']) & set(all_universities_associations_valence['cue_word']) & set(gpt_4o_associations_valence['cue_word']) & set(cue_words_students) & set(translated))

['fisica', 'matematica', 'esame']